In [30]:
#Cell 1 - Download requirements
!pip install datasets anthropic requests python-dotenv matplotlib -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# ── Cell 2 — Configuration ────────────────────────────────────────────────────
from dotenv import load_dotenv
import os, anthropic

MODEL      = "claude-sonnet-4-6"
MAX_TOKENS = 20000
DATASET    = "princeton-nlp/SWE-bench_Verified"

BATCH = 1   # 0 = first batch, 1 = second batch, etc.

# Sonnet 4.6 pricing (per million tokens)
INPUT_COST_PER_M  = 3.00
OUTPUT_COST_PER_M = 15.00

SHARED_INSTANCES_FILE = "pilot_instances.json"
PREDICTIONS_FILE      = f"predictions_baseline_batch{BATCH}.jsonl"
EXTRACTED_FILE        = f"predictions_baseline_batch{BATCH}_extracted.jsonl"
FIXED_FILE            = f"predictions_baseline_batch{BATCH}_fixed.jsonl"
RESULTS_FILE          = f"results_baseline_batch{BATCH}.json"

RUN_ID_BASE           = f"baseline_batch{BATCH}"

load_dotenv(override=True)
client = anthropic.Anthropic()


In [32]:
# ── Cell 3 — Load dataset + pilot instances ───────────────────────────────────
import json
from datasets import load_dataset

ds        = load_dataset(DATASET, split="test")
ds_lookup = {inst["instance_id"]: inst for inst in ds}

with open(SHARED_INSTANCES_FILE) as f:
    all_batches = json.load(f)  # list of batches, each a list of instance_ids

pilot_instances = [ds_lookup[iid] for iid in all_batches[BATCH] if iid in ds_lookup]
inst_lookup     = {i["instance_id"]: i for i in pilot_instances}

print(f"Batch {BATCH}: {len(pilot_instances)} instances")


Batch 1: 50 instances


In [33]:
# ── Cell 4 — Repo utilities ───────────────────────────────────────────────────
import subprocess, os

# Root folder where all repos are cloned, e.g. repos/django, repos/astropy
REPOS_BASE = "repos"

# Maps the org prefix of an instance_id to its local clone folder name
# e.g. "django__django-11333" → repos/django
ORG_TO_DIR = {
    "astropy"     : "astropy",
    "django"      : "django",
    "matplotlib"  : "matplotlib",
    "pylint-dev"  : "pylint",
    "pytest-dev"  : "pytest",
    "scikit-learn": "scikit-learn",
    "sphinx-doc"  : "sphinx",
    "sympy"       : "sympy",
    "pydata"      : "xarray",
}

def get_repo_dir(iid):
    """Return the local clone path for an instance, e.g. 'repos/django'."""
    org = iid.split("__")[0]
    return os.path.join(REPOS_BASE, ORG_TO_DIR.get(org, org))

def get_file_tree(repo_dir, commit):
    """Return all file paths in the repo at a given commit as a newline-separated string.
    Used by the localization step to show the model the full file tree."""
    r = subprocess.run(
        ["git", "ls-tree", "-r", "--name-only", commit],
        capture_output=True, text=True, cwd=repo_dir, encoding="utf-8",
    )
    return r.stdout.strip() if r.returncode == 0 else ""

def get_file_content(repo_dir, commit, filepath):
    """Return the content of a single file at a given commit, or None if it doesn't exist.
    Used to fetch source files for the patch generation prompt."""
    r = subprocess.run(
        ["git", "show", f"{commit}:{filepath}"],
        capture_output=True, text=True, cwd=repo_dir, encoding="utf-8", errors="replace",
    )
    return r.stdout.replace("\r\n", "\n").replace("\r", "\n") if r.returncode == 0 else None

print("Repo utilities ready")


Repo utilities ready


In [34]:
# ── Cell 5 — Cost calculator ──────────────────────────────────────────────────

def compute_cost(usage):
    """Convert a response's usage object into token counts and USD cost."""
    input_cost  = usage.input_tokens  / 1_000_000 * INPUT_COST_PER_M
    output_cost = usage.output_tokens / 1_000_000 * OUTPUT_COST_PER_M
    return {
        "input_tokens"   : usage.input_tokens,
        "output_tokens"  : usage.output_tokens,
        "input_cost_usd" : round(input_cost,  6),
        "output_cost_usd": round(output_cost, 6),
        "total_cost_usd" : round(input_cost + output_cost, 6),
    }


In [35]:
# ── Cell 6 — File localization ────────────────────────────────────────────────
import json as _json, re as _re

LOCALIZE_SYSTEM = (
    "You are a software engineer. Given a GitHub issue and the repository file tree, "
    "identify which files need to be modified to fix the issue.\n"
    "Return ONLY a JSON object: {\"files\": [\"path/to/file.py\", ...]}, max 21 files, most relevant first.\n"
    "Do not explain. Do not include any text before or after the JSON."
)

LOCALIZE_RETRY_SYSTEM = (
    "You are a software engineer. Output ONLY raw JSON — no explanation, no markdown, no prose.\n"
    "Example of the ONLY acceptable response format:\n"
    "{\"files\": [\"django/db/models/query.py\", \"django/db/models/sql/compiler.py\"]}\n"
    "Now identify which files need to be modified to fix the following issue."
)


def parse_files_from_response(text):
    """Extract file list from model response — handles JSON, fenced JSON, and prose fallback."""
    try:
        return _json.loads(text).get("files", [])
    except Exception:
        pass
    cleaned = _re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    try:
        return _json.loads(cleaned).get("files", [])
    except Exception:
        pass
    return _re.findall(r'"([^"\s]+\.[a-z]+)"', text)


def localize(inst):
    """
    Identify which files need to be edited for a given instance.
    1. Ask the model given the issue + file tree.
    2. Validate each path exists at base_commit.
    3. On failure: retry once with a stricter prompt.
    Returns (valid_files, cost, tag).
    """
    bc       = inst["base_commit"]
    repo_dir = get_repo_dir(inst["instance_id"])
    tree     = get_file_tree(repo_dir, bc)
    user_msg = (
        f"<issue>\n{inst['problem_statement']}\n</issue>\n\n"
        f"<file_tree>\n{tree}\n</file_tree>"
    )

    # ── First attempt ─────────────────────────────────────────────────────────
    resp  = client.messages.create(
        model=MODEL, max_tokens=256, temperature=0.0,
        system=LOCALIZE_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    files = parse_files_from_response(resp.content[0].text.strip())
    valid = [fp for fp in files[:4] if get_file_content(repo_dir, bc, fp) is not None]
    cost  = compute_cost(resp.usage)["total_cost_usd"]

    if valid:
        return valid, cost, ""

    # ── Retry with stricter prompt ─────────────────────────────────────────────
    resp2  = client.messages.create(
        model=MODEL, max_tokens=256, temperature=0.0,
        system=LOCALIZE_RETRY_SYSTEM,
        messages=[{"role": "user", "content": user_msg}],
    )
    files2 = parse_files_from_response(resp2.content[0].text.strip())
    valid  = [fp for fp in files2[:4] if get_file_content(repo_dir, bc, fp) is not None]
    cost  += compute_cost(resp2.usage)["total_cost_usd"]

    return valid, cost, " (retry)"


# ── Run localization over all pilot instances ──────────────────────────────────
localized      = {}
total_loc_cost = 0.0

for i, inst in enumerate(pilot_instances):
    iid      = inst["instance_id"]
    repo_dir = get_repo_dir(iid)

    if not os.path.isdir(repo_dir):
        localized[iid] = []
        print(f"[{i+1}/{len(pilot_instances)}] {iid}: repo not found at {repo_dir}")
        continue

    valid, cost, tag = localize(inst)
    localized[iid]   = valid
    total_loc_cost  += cost

    if valid:
        print(f"[{i+1}/{len(pilot_instances)}] {iid}: {valid}{tag}  ${cost:.4f}")
    else:
        print(f"[{i+1}/{len(pilot_instances)}] {iid}: no files found{tag}")

print(f"\nLocalization done. Total cost: ${total_loc_cost:.4f}")


[1/50] sympy__sympy-13877: ['sympy/matrices/matrices.py']  $0.0654
[2/50] django__django-12741: ['django/db/backends/base/operations.py', 'django/core/management/commands/flush.py', 'django/test/utils.py', 'django/db/backends/mysql/operations.py']  $0.3059
[3/50] matplotlib__matplotlib-23412: ['lib/matplotlib/patches.py', 'lib/matplotlib/backends/backend_ps.py', 'lib/matplotlib/backends/backend_pdf.py', 'lib/matplotlib/backends/backend_svg.py']  $0.2837
[4/50] sphinx-doc__sphinx-9602: ['sphinx/domains/python.py']  $0.0693
[5/50] pylint-dev__pylint-8898: ['pylint/config/argument.py']  $0.2010
[6/50] django__django-14534: ['django/forms/boundfield.py']  $0.3198
[7/50] django__django-15732: ['django/db/backends/base/schema.py', 'django/db/backends/postgresql/schema.py']  $0.3268
[8/50] matplotlib__matplotlib-14623: ['lib/matplotlib/scale.py', 'lib/matplotlib/axes/_base.py']  $0.2746
[9/50] django__django-14376: ['django/db/backends/mysql/base.py']  $0.3182
[10/50] sympy__sympy-16792: ['sy

In [36]:
# ── Cell 7 — Localization accuracy ────────────────────────────────────────────
import re as _re

def gold_files(inst):
    """Files touched by the gold patch — used as ground truth for localization eval."""
    return set(_re.findall(r"^--- a/(.+)$", inst["patch"], _re.MULTILINE))

full_hit, partial, miss = [], [], []

for inst in pilot_instances:
    iid     = inst["instance_id"]
    gold    = gold_files(inst)
    found   = set(localized.get(iid, []))
    overlap = gold & found

    if overlap == gold:
        full_hit.append(iid)
    elif overlap:
        partial.append(iid)
    else:
        miss.append(iid)

n = len(pilot_instances)
print(f"Full    (all gold files found) : {len(full_hit)}/{n}  ({100*len(full_hit)/n:.0f}%)")
print(f"Partial (>=1 gold file found)  : {len(partial)}/{n}  ({100*len(partial)/n:.0f}%)")
print(f"Miss    (no gold files found)  : {len(miss)}/{n}  ({100*len(miss)/n:.0f}%)")

if partial:
    print("\nPartial:")
    for iid in partial:
        gold  = gold_files(inst_lookup[iid])
        found = set(localized[iid])
        print(f"  {iid}")
        print(f"    found : {sorted(found)}")
        print(f"    missed: {sorted(gold - found)}")

if miss:
    print("\nMiss:")
    for iid in miss:
        print(f"  {iid}: gold={sorted(gold_files(inst_lookup[iid]))}, got={localized.get(iid, [])}")


Full    (all gold files found) : 28/50  (56%)
Partial (>=1 gold file found)  : 18/50  (36%)
Miss    (no gold files found)  : 4/50  (8%)

Partial:
  sympy__sympy-13877
    found : ['sympy/matrices/matrices.py']
    missed: ['sympy/utilities/randtest.py']
  pylint-dev__pylint-8898
    found : ['pylint/config/argument.py']
    missed: ['pylint/utils/__init__.py', 'pylint/utils/utils.py']
  matplotlib__matplotlib-14623
    found : ['lib/matplotlib/axes/_base.py', 'lib/matplotlib/scale.py']
    missed: ['lib/matplotlib/ticker.py', 'lib/mpl_toolkits/mplot3d/axes3d.py']
  django__django-14376
    found : ['django/db/backends/mysql/base.py']
    missed: ['django/db/backends/mysql/client.py']
  astropy__astropy-14369
    found : ['astropy/io/ascii/cds.py', 'astropy/io/ascii/mrt.py', 'astropy/io/ascii/tests/test_cds.py', 'astropy/units/format/cds.py']
    missed: ['astropy/units/format/cds_parsetab.py']
  matplotlib__matplotlib-25775
    found : ['lib/matplotlib/backend_bases.py', 'lib/matplotli

In [37]:
# ── Cell 8 — Prompt builder ───────────────────────────────────────────────────

def _format_file(content, filepath):
    """Add line numbers to file content so the model can reference exact lines in @@ headers."""
    numbered = "\n".join(f"{i+1} {line}" for i, line in enumerate(content.split("\n")))
    return f"[start of {filepath}]\n{numbered}\n[end of {filepath}]"

_PATCH_RULES = (
    "PATCH RULES:\n"
    "1. Context lines must be copied CHARACTER-FOR-CHARACTER from the snippet, "
    "including every leading space — do not paraphrase or change indentation.\n"
    "2. Use the line numbers shown at the start of each snippet line to set "
    "the correct @@ -N,M +N,M header.\n"
    "3. Never start or end a hunk with a blank context line — anchor on the "
    "nearest non-blank line above and below your change.\n"
    "4. Keep changes minimal. Do not add imports already present in the file.\n"
)

_PATCH_FORMAT = (
    "<patch>\n"
    "--- a/file.py\n"
    "+++ b/file.py\n"
    "@@ -1,27 +1,35 @@\n"
    " def euclidean(a, b):\n"
    "-    while b:\n"
    "-        a, b = b, a % b\n"
    "-    return a\n"
    "+    if b == 0:\n"
    "+        return a\n"
    "+    return euclidean(b, a % b)\n"
    "</patch>\n"
)

def build_prompt(inst):
    """Build the patch generation prompt for a single instance.
    Fetches the localized files at base_commit and formats them with line numbers."""
    repo_dir = get_repo_dir(inst["instance_id"])
    bc       = inst["base_commit"]
    files    = localized.get(inst["instance_id"], [])

    code_blocks = []
    for fp in files:
        content = get_file_content(repo_dir, bc, fp)
        if content is not None:
            code_blocks.append(_format_file(content, fp))

    code_section = "\n\n".join(code_blocks) if code_blocks else "(no files retrieved)"

    return (
        "You will be provided with a partial code base and an issue statement "
        "explaining a problem to resolve.\n"
        f"<issue>\n{inst['problem_statement']}\n</issue>\n"
        f"<code>\n{code_section}\n</code>\n"
        f"{_PATCH_RULES}\n"
        "Output your patch in the following format:\n"
        f"{_PATCH_FORMAT}"
    )


In [38]:
# ── Cell 9 — Generate patches ─────────────────────────────────────────────────
import json
from pathlib import Path

results = []
out = Path(PREDICTIONS_FILE)
out.unlink(missing_ok=True)  # start fresh for this batch

for i, inst in enumerate(pilot_instances):
    iid = inst["instance_id"]
    print(f"[{i+1}/{len(pilot_instances)}] {iid} ... ", end="", flush=True)

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            temperature=0.0,
            messages=[{"role": "user", "content": build_prompt(inst)}],
        )
        cost   = compute_cost(response.usage)
        record = {
            "instance_id"        : iid,
            "model_name_or_path" : MODEL,
            "raw_response"       : response.content[0].text,
            "repo"               : inst["repo"],
            **cost,
        }
        print(f"${record['total_cost_usd']:.5f}  ({record['input_tokens']} in / {record['output_tokens']} out)")

    except Exception as e:
        print(f"ERROR: {e}")
        record = {
            "instance_id"        : iid,
            "model_name_or_path" : MODEL,
            "raw_response"       : "",
            "repo"               : inst["repo"],
            "error"              : str(e),
            "input_tokens": 0, "output_tokens": 0,
            "input_cost_usd": 0, "output_cost_usd": 0, "total_cost_usd": 0,
        }

    results.append(record)
    with open(out, "a") as f:
        f.write(json.dumps(record) + "\n")

total_cost = sum(r["total_cost_usd"] for r in results)
errors     = [r["instance_id"] for r in results if r.get("error")]
print(f"\nDone. Total cost: ${total_cost:.4f}")
if errors:
    print(f"Errors ({len(errors)}): {errors}")


[1/50] sympy__sympy-13877 ... $0.16632  (53125 in / 463 out)
[2/50] django__django-12741 ... $0.07966  (23507 in / 609 out)
[3/50] matplotlib__matplotlib-23412 ... $0.39268  (129083 in / 362 out)
[4/50] sphinx-doc__sphinx-9602 ... $0.22160  (18591 in / 11055 out)
[5/50] pylint-dev__pylint-8898 ... $0.02780  (6335 in / 586 out)
[6/50] django__django-14534 ... $0.01401  (4035 in / 127 out)
[7/50] django__django-15732 ... $0.09362  (25028 in / 1236 out)
[8/50] matplotlib__matplotlib-14623 ... $0.20661  (62526 in / 1269 out)
[9/50] django__django-14376 ... $0.01972  (5585 in / 198 out)
[10/50] sympy__sympy-16792 ... $0.21785  (70161 in / 491 out)
[11/50] django__django-15695 ... $0.28113  (85896 in / 1563 out)
[12/50] sympy__sympy-21930 ... $0.28310  (91386 in / 596 out)
[13/50] django__django-15957 ... $0.22873  (60843 in / 3080 out)
[14/50] sympy__sympy-17318 ... $0.09186  (27530 in / 618 out)
[15/50] astropy__astropy-14369 ... $0.09068  (26643 in / 717 out)
[16/50] django__django-13820 

In [39]:
# ── Cell 10 — Extract patches ─────────────────────────────────────────────────
import json, re

def extract_best_patch(raw_response):
    """
    Extract the model's final patch from its raw response.
    Takes the last <patch> block per file — the model often outputs
    exploratory drafts before settling on its final answer.
    """
    patches = re.findall(r"<patch>(.*?)</patch>", raw_response, re.DOTALL)
    valid   = [p.strip() for p in patches if p.strip().startswith(("diff --git", "--- a/"))]
    if not valid:
        return ""

    # Track which patch index last touches each file
    file_to_idx = {}
    for idx, p in enumerate(valid):
        for f in re.findall(r"^--- a/(\S+)", p, re.MULTILINE):
            file_to_idx[f] = idx

    last_idx   = max(file_to_idx.values()) if file_to_idx else len(valid) - 1
    last_patch = valid[last_idx]
    last_files = set(re.findall(r"^--- a/(\S+)", last_patch, re.MULTILINE))

    if last_files == set(file_to_idx.keys()):
        return last_patch + "\n"

    # Some files only appear in earlier patches — combine, taking last version per file
    seen, combined = set(), []
    for p in reversed(valid):
        for filepath, hunks in _parse_diff_blocks(p):
            if filepath not in seen:
                seen.add(filepath)
                combined.insert(0, (filepath, hunks))

    out = []
    for filepath, hunks in combined:
        out.append(f"--- a/{filepath}")
        out.append(f"+++ b/{filepath}")
        for hdr, body in hunks:
            out.append(hdr)
            out.extend(body)
    return "\n".join(out) + "\n"


def _parse_diff_blocks(patch_text):
    """Parse a unified diff into [(filepath, [(hunk_header, hunk_body)])]."""
    files = []
    cur_file = cur_hdr = None
    cur_hunks = []
    cur_body  = []
    for line in patch_text.split("\n"):
        if line.startswith("--- a/"):
            if cur_file:
                if cur_hdr is not None:
                    cur_hunks.append((cur_hdr, cur_body))
                files.append((cur_file, cur_hunks))
            cur_file = line[6:]; cur_hunks = []; cur_hdr = None; cur_body = []
        elif line.startswith("+++ b/"):
            pass
        elif line.startswith("@@ "):
            if cur_hdr is not None:
                cur_hunks.append((cur_hdr, cur_body))
            cur_hdr = line; cur_body = []
        elif cur_hdr is not None and line and line[0] in " -+":
            cur_body.append(line)
    if cur_file:
        if cur_hdr is not None:
            cur_hunks.append((cur_hdr, cur_body))
        files.append((cur_file, cur_hunks))
    return files


with open(PREDICTIONS_FILE) as fin, open(EXTRACTED_FILE, "w") as fout:
    for line in fin:
        pred  = json.loads(line.strip())
        iid   = pred["instance_id"]
        patch = extract_best_patch(pred.get("raw_response", ""))
        fout.write(json.dumps({
            "instance_id"        : iid,
            "model_patch"        : patch,
            "model_name_or_path" : MODEL,
        }) + "\n")

print(f"Written → {EXTRACTED_FILE}")


Written → predictions_agentless_batch1_extracted.jsonl


In [40]:
# ── Cell 11 — Fix hunk line numbers ───────────────────────────────────────────
import json, re

def fix_hunk(hdr, body, file_lines):
    """
    Re-anchor a hunk by searching for its context+removed lines in the actual file.
    Also handles reversed patches and fixes wrong @@ counts.
    """
    def parse_ops(lines):
        ops = []
        for line in lines:
            if line.startswith("-") and not line.startswith("---"):
                ops.append(("-", line[1:]))
            elif line.startswith("+") and not line.startswith("+++"):
                ops.append(("+", line[1:]))
            else:
                ops.append((" ", line[1:] if line.startswith(" ") else line))
        return ops

    def search_in_file(ops, file_lines):
        """Return 1-based start line if context+removed lines are found in file."""
        needle = [c for op, c in ops if op in (" ", "-")]
        if not needle or not file_lines:
            return None
        fs = [l.rstrip() for l in file_lines]
        ss = [l.rstrip() for l in needle]
        for i in range(len(fs) - len(ss) + 1):
            if fs[i : i + len(ss)] == ss:
                return i + 1
        return None

    ops   = parse_ops(body)
    found = search_in_file(ops, file_lines)

    # Try reversed patch if forward search failed
    if found is None:
        flipped = [("+", c) if op == "-" else ("-", c) if op == "+" else (" ", c) for op, c in ops]
        found   = search_in_file(flipped, file_lines)
        if found is not None:
            ops = flipped

    # Fall back to original line number if context not found
    if found is None:
        m     = re.match(r"@@ -(\d+)", hdr)
        found = int(m.group(1)) if m else 1
    elif file_lines:
        # Replace context/removed lines with exact file content to fix whitespace
        offset, new_ops = 0, []
        for op, c in ops:
            if op in (" ", "-"):
                idx = found - 1 + offset
                new_ops.append((op, file_lines[idx] if idx < len(file_lines) else c))
                offset += 1
            else:
                new_ops.append((op, c))
        ops = new_ops

    orig_n  = sum(1 for op, _ in ops if op in (" ", "-"))
    new_n   = sum(1 for op, _ in ops if op in (" ", "+"))
    ctx     = re.search(r"@@[^@]*@@\s*(.*)", hdr)
    new_hdr = f"@@ -{found},{orig_n} +{found},{new_n} @@" + (f" {ctx.group(1)}" if ctx and ctx.group(1) else "")
    new_body = [f"{op}{c}" if op != " " else f" {c}" for op, c in ops]
    return new_hdr, new_body


def fix_patch_line_numbers(patch, iid, base_commit):
    """Fix @@ headers in every hunk using the real file content at base_commit."""
    repo_dir = get_repo_dir(iid)
    if not os.path.isdir(repo_dir):
        return patch
    blocks = _parse_diff_blocks(patch)
    if not blocks:
        return patch
    out = []
    for filepath, hunks in blocks:
        content    = get_file_content(repo_dir, base_commit, filepath)
        file_lines = content.splitlines() if content else None
        fixed_hunks = [fix_hunk(hdr, body, file_lines) if file_lines else (hdr, body) for hdr, body in hunks]
        # Sort hunks by start line — model sometimes outputs them out of order
        fixed_hunks.sort(key=lambda hb: int(re.match(r"@@ -(\d+)", hb[0]).group(1)) if re.match(r"@@ -(\d+)", hb[0]) else 0)
        out += [f"--- a/{filepath}", f"+++ b/{filepath}"]
        for hdr, body in fixed_hunks:
            out.append(hdr)
            out.extend(body)
    return "\n".join(out) + "\n"


fixed_count = 0
with open(EXTRACTED_FILE) as fin, open(FIXED_FILE, "w") as fout:
    for line in fin:
        pred  = json.loads(line.strip())
        iid   = pred["instance_id"]
        patch = pred.get("model_patch", "")
        if patch and iid in inst_lookup:
            fixed = fix_patch_line_numbers(patch, iid, inst_lookup[iid]["base_commit"])
            if fixed != patch:
                fixed_count += 1
            patch = fixed
        fout.write(json.dumps({
            "instance_id"        : iid,
            "model_patch"        : patch,
            "model_name_or_path" : MODEL,
        }) + "\n")

print(f"Fixed {fixed_count} patches → {FIXED_FILE}")


Fixed 40 patches → predictions_agentless_batch1_fixed.jsonl


In [41]:
# ── Cell 12 — Evaluate ────────────────────────────────────────────────────────
import subprocess, json, os

VENV = "/tmp/swebench_venv"

def to_wsl_path(path):
    """Convert a Windows absolute path to a WSL /mnt/ path."""
    cwd   = os.getcwd()
    drive = cwd[0].lower()
    return "/mnt/" + drive + "/" + cwd[3:].replace(os.sep, "/") + "/" + path

def wsl(args, label=""):
    if label:
        print(f"  [{label}]")
    r = subprocess.run(
        ["wsl", "-d", "Ubuntu"] + args,
        capture_output=True, text=True, encoding="utf-8", errors="replace"
    )
    if r.stdout: print(r.stdout[-3000:])
    if r.stderr: print(r.stderr[-1000:])
    return r.returncode

def evaluate(predictions_file, run_id):
    """Run SWE-bench evaluation on a predictions file."""
    with open(predictions_file) as f:
        instance_ids = [json.loads(l)["instance_id"] for l in f]
    wsl([
        f"{VENV}/bin/python", "-m", "swebench.harness.run_evaluation",
        "--dataset_name", DATASET,
        "--predictions_path", to_wsl_path(predictions_file),
        "--instance_ids", *instance_ids,
        "--max_workers", "16",
        "--run_id", run_id,
    ], "evaluate")

print("Setting up venv...")
wsl(["python3", "-m", "venv", VENV], "venv")
wsl([f"{VENV}/bin/pip", "install", "swebench", "-q"], "pip install")

print(f"\n── Extracted predictions ──────────────────────────────────")
evaluate(EXTRACTED_FILE, f"{RUN_ID_BASE}_extracted")

print(f"\n── Fixed predictions ──────────────────────────────────────")
evaluate(FIXED_FILE, f"{RUN_ID_BASE}_fixed")

print("\nEvaluation complete.")


Setting up venv...
  [venv]
  [pip install]

── Extracted predictions ──────────────────────────────────
  [evaluate]
__.py
patch: **** malformed patch at line 10:                  else:


Check (logs/run_evaluation/agentless_batch1_extracted/claude-sonnet-4-6/sphinx-doc__sphinx-8548/run_instance.log) for more information.
sphinx-doc__sphinx-8551: >>>>> Patch Apply Failed:
patch: **** malformed patch at line 18:              # NOTE: searching for exact match, object type is not considered

patching file sphinx/domains/python.py

Check (logs/run_evaluation/agentless_batch1_extracted/claude-sonnet-4-6/sphinx-doc__sphinx-8551/run_instance.log) for more information.
sphinx-doc__sphinx-11510: >>>>> Patch Apply Failed:
patch: **** malformed patch at line 24:          self.env.note_included(filename)

patching file sphinx/directives/other.py

Check (logs/run_evaluation/agentless_batch1_extracted/claude-sonnet-4-6/sphinx-doc__sphinx-11510/run_instance.log) for more information.
sphinx-doc__sph

In [42]:
# ── Cell 13 — Parse results ───────────────────────────────────────────────────
import json, glob, os

def count_files(inst):
    return sum(1 for line in inst["patch"].splitlines() if line.startswith("--- a/"))

def load_report(run_id):
    path = f"{MODEL}.{RUN_ID_BASE}_{run_id}.json"
    if not os.path.exists(path):
        print(f"Not found: {path}")
        return None
    with open(path) as f:
        return json.load(f)

def print_report(report, label):
    if report is None:
        return
    total      = report["total_instances"]
    resolved   = report["resolved_instances"]
    errors     = report["error_instances"]
    resolved_ids    = set(report.get("resolved_ids", []))
    resolved_single = [iid for iid in resolved_ids if count_files(inst_lookup[iid]) == 1 if iid in inst_lookup]
    resolved_multi  = [iid for iid in resolved_ids if count_files(inst_lookup[iid]) >  1 if iid in inst_lookup]
    total_single    = [i for i in pilot_instances if count_files(i) == 1]
    total_multi     = [i for i in pilot_instances if count_files(i) >  1]

    print(f"{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Overall    : {resolved:>3} / {total}  ({100*resolved/total:.1f}%)")
    print(f"  Single file: {len(resolved_single):>3} / {len(total_single)}  ({100*len(resolved_single)/len(total_single):.1f}%)" if total_single else "  Single file: n/a")
    print(f"  Multi file : {len(resolved_multi):>3} / {len(total_multi)}  ({100*len(resolved_multi)/len(total_multi):.1f}%)" if total_multi else "  Multi file : n/a")
    print(f"  Errors     : {errors:>3} / {total}")
    print(f"{'='*50}")

    summary = {
        "notebook": "baseline", "model": MODEL, "batch": BATCH, "pipeline": label,
        "total": total, "resolved": resolved, "errors": errors,
        "resolve_rate": resolved / total if total else 0,
        "single_file": {"total": len(total_single), "resolved": len(resolved_single)},
        "multi_file":  {"total": len(total_multi),  "resolved": len(resolved_multi)},
    }
    out = RESULTS_FILE.replace(".json", f"_{label}.json")
    with open(out, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"  Saved → {out}")

print(f"Available reports: {glob.glob('*.json')}\n")
print_report(load_report("extracted"), "extracted")
print()
print_report(load_report("fixed"),     "fixed")


Available reports: ['claude-sonnet-4-6.agentless_batch0_clean.json', 'claude-sonnet-4-6.agentless_batch0_extracted.json', 'claude-sonnet-4-6.agentless_batch0_fixed.json', 'claude-sonnet-4-6.agentless_batch1_extracted.json', 'claude-sonnet-4-6.agentless_batch1_fixed.json', 'claude-sonnet-4-6.ast_flowgraph_batch0_extracted.json', 'claude-sonnet-4-6.ast_flowgraph_batch0_fixed.json', 'claude-sonnet-4-6.ast_graph_batch0_extracted.json', 'claude-sonnet-4-6.ast_graph_batch0_fixed.json', 'pilot_instances.json', 'results_agentless_batch0.json', 'results_agentless_batch0_extracted.json', 'results_agentless_batch0_fixed.json', 'results_ast_batch0_extracted.json', 'results_ast_batch0_fixed.json', 'results_ast_flowgraph_batch0_extracted.json', 'results_ast_flowgraph_batch0_fixed.json', 'results_ast_graph_batch0_extracted.json', 'results_ast_graph_batch0_fixed.json']

  extracted
  Overall    :  12 / 50  (24.0%)
  Single file:   7 / 25  (28.0%)
  Multi file :   5 / 25  (20.0%)
  Errors     :  24 / 5

In [43]:
# ── Cell 14 — Localization coverage vs eval results ──────────────────────────
import re as _re

# Load the fixed report (best results)
report = load_report("fixed")
if report is None:
    print("Run cell 13 first.")
else:
    resolved_ids = set(report.get("resolved_ids", []))

    rows = []
    for inst in pilot_instances:
        iid        = inst["instance_id"]
        gold       = set(_re.findall(r"^--- a/(.+)$", inst["patch"], _re.MULTILINE))
        found      = set(localized.get(iid, []))
        overlap    = gold & found
        coverage   = "full" if overlap == gold else "partial" if overlap else "miss"
        rows.append({"iid": iid, "passed": iid in resolved_ids, "coverage": coverage,
                     "gold": sorted(gold), "found": sorted(found), "missed": sorted(gold - found)})

    print(f"{'Coverage':<10}  {'Total':>6}  {'Passed':>8}  {'Failed':>8}  {'Pass rate':>10}")
    print("-" * 48)
    for cov in ("full", "partial", "miss"):
        subset = [r for r in rows if r["coverage"] == cov]
        passed = sum(r["passed"] for r in subset)
        rate   = f"{100*passed/len(subset):.0f}%" if subset else "—"
        print(f"{cov:<10}  {len(subset):>6}  {passed:>8}  {len(subset)-passed:>8}  {rate:>10}")
    print("-" * 48)
    total_pass = sum(r["passed"] for r in rows)
    print(f"{'total':<10}  {len(rows):>6}  {total_pass:>8}  {len(rows)-total_pass:>8}  {100*total_pass/len(rows):.0f}%")

    full_failed = [r for r in rows if r["coverage"] == "full" and not r["passed"]]
    if full_failed:
        print(f"\nFull coverage but FAILED ({len(full_failed)}) — patch generation issue:")
        for r in full_failed:
            print(f"  {r['iid']}")

    lucky = [r for r in rows if r["coverage"] != "full" and r["passed"]]
    if lucky:
        print(f"\nIncomplete coverage but PASSED ({len(lucky)}) — got lucky:")
        for r in lucky:
            print(f"  {r['iid']}  [{r['coverage']}]  missed: {r['missed']}")


Coverage     Total    Passed    Failed   Pass rate
------------------------------------------------
full            28        17        11         61%
partial         18         6        12         33%
miss             4         0         4          0%
------------------------------------------------
total           50        23        27  46%

Full coverage but FAILED (11) — patch generation issue:
  sphinx-doc__sphinx-9602
  django__django-15732
  django__django-15695
  sympy__sympy-21930
  sphinx-doc__sphinx-11510
  sympy__sympy-14248
  django__django-16631
  django__django-13023
  astropy__astropy-13453
  django__django-13809
  sympy__sympy-12489

Incomplete coverage but PASSED (6) — got lucky:
  sympy__sympy-13877  [partial]  missed: ['sympy/utilities/randtest.py']
  matplotlib__matplotlib-25775  [partial]  missed: ['lib/matplotlib/backends/backend_agg.py', 'lib/matplotlib/backends/backend_cairo.py']
  pytest-dev__pytest-8399  [partial]  missed: ['src/_pytest/python.py']
  pydata_